In [ ]:
import cv2 #cv2/open cv 影像讀取、增強、切割與儲存
import os
from pathlib import Path #pathlib 處理檔案路徑

In [ ]:
def setup_folders():
    folders = ['preprocessing_input', 'preprocessing_output', 'preprocessing_check']
    #preprocessing_input 原圖檔輸入, preprocessing_output 完成分切的圖, preprocessed_check 完成對比度調整的圖

    for f in folders: #依序抓folders中的字串
        Path(f).mkdir(parents=True, exist_ok=True)
        #Path(f)將抓到的字串轉成一個路徑物件, .mkdir用前面轉換的路徑建立一個資料夾
        #parents=True偵測未建立的父目錄，未建立就自動建立 exist_ok=True偵測已建立的資料夾就跳過

    print(f"Folder has been created: {folders}")
    

In [ ]:
def process_images(tile_size=256, overlap=30):
    input_folder = 'preprocessing_input'
    output_tile_folder = 'preprocessing_output'
    full_output_folder = 'preprocessing_check'

    image_extensions = ['.jpg', '.jpeg', '.tif'] #列出可讀取的副檔名, 排除掉不可讀取的檔案
    folders = [input_folder, output_tile_folder, full_output_folder]

    for folder in folders:
        if not os.path.exists(folder): #確認資料夾存在. os(Operating System).pathy在作業系統中的路徑, .exists存在與否(回傳true or false)
            print(f"{folder} not exist")
            return

    image_files = [f for f in os.listdir(input_folder)
                   #f(列入list的結果)for f(前面f依據此for f遍歷檔案夾的結果) #listdir讀取資料夾內容物(回傳檔名), 所以最前面的f只接收檔名
                   if os.path.splitext(f)[1].lower() in image_extensions]
                   #os.path.splitext(f)遇到第一個"."就分切路徑(這邊正好是檔名) #[1].lower()強制副檔名小寫

    if not image_files:
        print("image not exist")
        return

    print(f"total image : {len(image_files)}")

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    #cv2.createCLAHE用OpenCV建立clahe處理器 #clipLimit=3.0這裡clahe偵測並先計算影像直方圖的平均, 3.0為設定最高為平均的3.0倍, 多出的部分被均質給每個點
    #tileGridSize=(8,8)虛擬拆解圖片去讀取並進行clipLimit=3.0處理, 並指定(column number, row number)

    for image_name in image_files:
        image_path = os.path.join(input_folder, image_name) #合成路徑
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        #cv2.imread讀取圖片至記憶體並透過header解碼 #cv2.IMREAD_GRAYSCALE是一個flag, 常數值為0, 用Gray = 0.299R + 0.587G + 0.114B公式轉灰階
        if img is None: continue #讀取失敗(ex.檔案毀損)則跳過

        denoised_img = cv2.bilateralFilter(img, 5, 75, 75)
        #denoised = cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace)
        #src 圖片原始數據 #d 5代表以目標像素為中心考慮直徑5像素的圓形區域
        #sigmaColor 決定多大的差異會被視為邊緣(越大將噪效果越好, 同時保護邊緣效果變差)
        #sigmaSpace 標準差or擴散係數, 影響力隨距離衰減的速度(越大衰減越慢)

        enhanced = clahe.apply(denoised_img)

        cv2.imwrite(os.path.join(full_output_folder, f'enhanced_{image_name}'), enhanced)
        #cv2.imwrite將矩陣數據再轉回圖檔, 根據偵測到的image_name副檔名, cv2.imwrite(路徑, 要寫入的檔案)
        #os.path.join(資料夾名(檔名前的所有路徑), 檔名), f'enhanced_{image_name}' 順便修改檔名

        h, w = enhanced.shape #(h, w) = (row, col) = (1040, 1388)
        stride = tile_size - overlap
        count = 0

        for y in range(0, h - tile_size + 1, stride):
            for x in range(0, w - tile_size + 1, stride): #range(start, stop, step) stop為最後一次迴圈起點邊界, 非實際終點位置
                tile = enhanced[y:y+tile_size, x:x+tile_size] #tile切出來的小圖的matrix #原始矩陣名[y:y+a(row範圍, y至y+a), x:x+b(col範圍)]
                tile_filename = f"{os.path.splitext(image_name)[0]}_y{y}_x{x}.jpg"
                cv2.imwrite(os.path.join(output_tile_folder, tile_filename), tile)
                count += 1

        print(f"{image_name}: {count} (tiles)")

In [ ]:
if __name__ == "__main__":
    setup_folders()
    process_images()